In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences #function used to make all text sequences the same length.
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout  #Converts word indexes into dense vector representations.

In [ ]:

# Load Dataset (Auto Download)

vocab_size = 5000   #Keep only the top 5000 most frequent UNIQUE words from the dataset.
max_length = 200

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocab_size)
# from Dataset - Binary labels (0 = negative, 1 = positive)

In [15]:
# -----------------------------
# Padding Sequences
# -----------------------------
X_train = pad_sequences(X_train, maxlen=max_length)
X_test = pad_sequences(X_test, maxlen=max_length)

In [ ]:
# -----------------------------
# Build LSTM Model
# -----------------------------
model = Sequential([
    Embedding(vocab_size, 128), # Embedding converts each number into a 128-dimensional vector.
    LSTM(128), # Output vector size = 128(neuron)
    Dropout(0.5),
    Dense(1, activation='sigmoid') # We have only 2 classes, need probability output, sigmoid is best for binary classification 
    # Sigmoid gives output value as 0 or 1
])

#This prepares model for training.
model.compile(
    loss='binary_crossentropy', #Loss measures how wrong the model is. calculates error between actual and predict
    optimizer='adam',  # updates weights to reduce loss. Automatically adjusts weights
    metrics=['accuracy']
)



In [ ]:
# -----------------------------
# Train Model
# -----------------------------
model.fit(
    X_train,
    y_train,
    epochs=5, #The model sees the entire training dataset once.
    batch_size=64, # Model processes 64 reviews at a time
    validation_data=(X_test, y_test)
)


Epoch 1/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 221s 560ms/step - accuracy: 0.6465 - loss: 0.6179 - val_accuracy: 0.8612 - val_loss: 0.3353
Epoch 2/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 245s 628ms/step - accuracy: 0.8779 - loss: 0.3050 - val_accuracy: 0.8564 - val_loss: 0.3350
Epoch 3/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 237s 563ms/step - accuracy: 0.9023 - loss: 0.2459 - val_accuracy: 0.8726 - val_loss: 0.3147
Epoch 4/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 255s 545ms/step - accuracy: 0.9203 - loss: 0.2172 - val_accuracy: 0.8723 - val_loss: 0.3292
Epoch 5/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 264s 549ms/step - accuracy: 0.9393 - loss: 0.1615 - val_accuracy: 0.8716 - val_loss: 0.3581


In [20]:

# -----------------------------
# Evaluate Model
# -----------------------------
loss, accuracy = model.evaluate(X_test, y_test)
print("\nTest Accuracy:", accuracy)

782/782 ━━━━━━━━━━━━━━━━━━━━ 74s 94ms/step - accuracy: 0.8680 - loss: 0.3688

Test Accuracy: 0.8715599775314331


In [ ]:
# -----------------------------
# Get Word Index for Decoding
# -----------------------------

# Understand what number corresponds to which word
# Decode reviews back into readable text
word_index = imdb.get_word_index()      


#Special Tokens
# Reverse dictionary (number → word)
# To convert numbers back to readable text.
reverse_word_index = {value+3: key for key, value in word_index.items()}
reverse_word_index[0] = "<PAD>"  #Padding value
reverse_word_index[1] = "<START>" #Start of review
reverse_word_index[2] = "<UNK>"    #Unknown word , Out-of-vocabulary
reverse_word_index[3] = "<UNUSED>"   #Reserved , Internal use

In [23]:
# -----------------------------
# Prediction Function
# -----------------------------
def predict_sentiment(text):
    words = text.lower().split()

    encoded = []
    for word in words:
        if word in word_index and word_index[word] < vocab_size:
            encoded.append(word_index[word] + 3)
        else:
            encoded.append(2)  # <UNK>

    padded = pad_sequences([encoded], maxlen=max_length)
    prediction = model.predict(padded)


    if prediction[0][0] > 0.5:
        return "Positive 😊"
    else:
        return "Negative 😞"

# -----------------------------
# User Input Loop
# -----------------------------
while True:
    user_input = input("\nEnter a movie review (type 'exit' to quit): ")

    if user_input.lower() == "exit":
        print("Program ended.")
        break

    result = predict_sentiment(user_input)
    print("Predicted Sentiment:", result)


Enter a movie review (type 'exit' to quit): Excellent direction and soundtrack.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
Predicted Sentiment: Positive 😊

Enter a movie review (type 'exit' to quit): Very entertaining and enjoyable.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
Predicted Sentiment: Positive 😊

Enter a movie review (type 'exit' to quit): Amazing movie with great acting.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
Predicted Sentiment: Positive 😊

Enter a movie review (type 'exit' to quit): Terrible movie and poor acting.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
Predicted Sentiment: Negative 😞

Enter a movie review (type 'exit' to quit): Very boring and predictable.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
Predicted Sentiment: Negative 😞

Enter a movie review (type 'exit' to quit): Waste of time.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
Predicted Sentiment: Negative 😞

Enter a movie review (type 'exit' to quit): exit
Program ended.
